# 02 — Data Exploration & Visualisation
**BraTS 2023 GLI Brain Tumor Segmentation | RCOEM 2026–27**

This notebook:
1. Loads preprocessed `.npz` files
2. Visualises multi-planar slices (axial, coronal, sagittal) for several patients
3. Analyses class imbalance across the full dataset
4. Shows 3D tumor sub-region overlays
5. Generates figures suitable for the project report

**Estimated time:** ~20 minutes

In [ ]:
import sys, os
PREPROCESSED_PATH = '/home/yourname/BraTS2023_Preprocessed'  # ← CHANGE THIS
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Preprocessed data: {PREPROCESSED_PATH}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from glob import glob

# Load all preprocessed file paths
npz_files = sorted(glob(f'{PREPROCESSED_PATH}/*.npz'))
print(f'Found {len(npz_files)} preprocessed patient files')
print(f'Example: {Path(npz_files[0]).name}')

## 🔍 Multi-Planar Visualisation — 3 Views for 1 Patient

In [ ]:
def plot_multiplanar(npz_path, modality_idx=0, modality_name='FLAIR'):
    data = np.load(npz_path)
    img  = data['image']  # (4, H, W, D)
    seg  = data['seg']    # (H, W, D)
    patient_id = Path(npz_path).stem

    vol = img[modality_idx]  # (H, W, D)
    H, W, D = vol.shape
    cmap = 'gray'

    label_colors = {1: '#e74c3c', 2: '#3498db', 3: '#2ecc71'}  # NCR=red, ED=blue, ET=green
    label_names  = {1: 'NCR', 2: 'ED', 3: 'ET'}

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Patient: {patient_id} | Modality: {modality_name}', fontsize=14, fontweight='bold')

    slices_views = [
        (vol[H//2, :, :],       seg[H//2, :, :],       'Sagittal (x=mid)'),
        (vol[:, W//2, :],       seg[:, W//2, :],       'Coronal (y=mid)'),
        (vol[:, :, D//2],       seg[:, :, D//2],       'Axial (z=mid)'),
    ]

    for col, (vol_slice, seg_slice, title) in enumerate(slices_views):
        # Raw modality
        axes[0, col].imshow(vol_slice.T, cmap=cmap, origin='lower')
        axes[0, col].set_title(f'{title}\n{modality_name} (normalised)')
        axes[0, col].axis('off')

        # With label overlay
        axes[1, col].imshow(vol_slice.T, cmap=cmap, origin='lower')
        for label_id, color in label_colors.items():
            mask = np.ma.masked_where(seg_slice != label_id, seg_slice)
            axes[1, col].imshow(mask.T, cmap=plt.cm.colors.ListedColormap([color]),
                                alpha=0.55, origin='lower')
        axes[1, col].set_title(f'{title}\nWith GT Labels')
        axes[1, col].axis('off')

    # Legend
    patches = [mpatches.Patch(color=c, label=f'Label {k}: {label_names[k]}')
               for k, c in label_colors.items()]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11,
               bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    save_name = f'multiplanar_{patient_id}.png'
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    return data

# Plot first patient
data0 = plot_multiplanar(npz_files[0])

## 📊 Class Imbalance Analysis (Full Dataset)

In [ ]:
# Load preprocessing report if available
report_path = Path(PREPROCESSED_PATH) / 'preprocessing_report.csv'
if report_path.exists():
    df = pd.read_csv(report_path)
    print(f'Loaded stats for {len(df)} patients from report')
else:
    # Compute on-the-fly (slower)
    print('No report found, computing stats...')
    records = []
    for npz in npz_files:
        data = np.load(npz)
        seg  = data['seg']
        u, c = np.unique(seg, return_counts=True)
        lc   = dict(zip(u.tolist(), c.tolist()))
        records.append({'patient_id': Path(npz).stem,
                        'ncr_voxels': lc.get(1, 0),
                        'ed_voxels':  lc.get(2, 0),
                        'et_voxels':  lc.get(3, 0)})
    df = pd.DataFrame(records)

# Class imbalance pie chart
total_ncr = df['ncr_voxels'].sum()
total_ed  = df['ed_voxels'].sum()
total_et  = df['et_voxels'].sum()
total_fg  = total_ncr + total_ed + total_et

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie: foreground composition
sizes  = [total_ncr, total_ed, total_et]
labels = ['NCR (Label 1)', 'Edema (Label 2)', 'ET (Label 3)']
colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Foreground Voxel Composition\n(all patients combined)', fontsize=12)

# Box plot: per-patient tumor sizes
box_data = [df['ncr_voxels']/1000, df['ed_voxels']/1000, df['et_voxels']/1000]
bp = axes[1].boxplot(box_data, patch_artist=True, notch=True,
                     medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_xticklabels(['NCR', 'Edema', 'ET'], fontsize=12)
axes[1].set_ylabel('Voxel count (thousands)', fontsize=11)
axes[1].set_title('Per-Patient Tumor Sub-Region Sizes', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('BraTS 2023 GLI — Class Imbalance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nForeground voxel breakdown:')
print(f'  NCR : {total_ncr:>12,} ({100*total_ncr/total_fg:.1f}%)')
print(f'  ED  : {total_ed:>12,} ({100*total_ed/total_fg:.1f}%)')
print(f'  ET  : {total_et:>12,} ({100*total_et/total_fg:.1f}%)')

## 🎨 Visualise Multiple Patients Side-by-Side

In [ ]:
# Show axial mid-slice for 8 random patients — all 4 modalities
import random
random.seed(42)
sample_files = random.sample(npz_files, min(8, len(npz_files)))

fig, axes = plt.subplots(8, 5, figsize=(20, 32))
modality_names = ['FLAIR', 'T1', 'T1ce', 'T2', 'GT Seg']

for row, npz_path in enumerate(sample_files):
    data = np.load(npz_path)
    img  = data['image']   # (4, H, W, D)
    seg  = data['seg']     # (H, W, D)
    mid_z = img.shape[3] // 2

    for col in range(4):
        axes[row, col].imshow(img[col, :, :, mid_z].T, cmap='gray', origin='lower')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(modality_names[col], fontsize=12, fontweight='bold')

    axes[row, 4].imshow(seg[:, :, mid_z].T, cmap='tab10', origin='lower', vmin=0, vmax=3)
    axes[row, 4].axis('off')
    if row == 0:
        axes[row, 4].set_title(modality_names[4], fontsize=12, fontweight='bold')
    axes[row, 0].set_ylabel(Path(npz_path).stem[:20], fontsize=7, rotation=0, labelpad=60)

plt.suptitle('BraTS 2023 GLI — Sample Patients (Axial Mid-Slice)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_patients_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n🏁 Notebook 02 complete. Proceed to 03_train_baseline.ipynb')